# Búsqueda de Hiperparámetros con MLflow

Registro de experimentos de búsqueda de hiperparámetros para el dataset de aerolíneas.

**Modelos:** Logistic Regression · KNN · Random Forest · XGBoost  
**Experimento MLflow:** `airline-satisfaction-hyperparam-search`

In [1]:
import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("mlflow").setLevel(logging.ERROR)

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import xgboost as xgb
from xgboost import XGBClassifier
import optuna
import mlflow
import mlflow.sklearn
import mlflow.xgboost

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42

In [2]:
# MLflow se configura via env var MLFLOW_TRACKING_URI=http://mlflow-proxy:5001
# Se puede verificar con:
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://mlflow-proxy:5001


## Carga de datos

In [3]:
TRAIN_PATH = "../datasets/aerolineas/train.csv"
TEST_PATH  = "../datasets/aerolineas/test.csv"

def load_dataset(path):
    df = pd.read_csv(path, index_col=0)
    df = df.dropna(subset=["Arrival Delay in Minutes"])
    return df

df_train = load_dataset(TRAIN_PATH)
df_test  = load_dataset(TEST_PATH)

print(f"Train: {df_train.shape} | Test: {df_test.shape}")

Train: (103594, 24) | Test: (25893, 24)


In [4]:
CATEGORICAL_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
TARGET_COL = "satisfaction"

def prepare_data(df, dummies=True):
    y = (df[TARGET_COL] == "satisfied").astype(int)
    X = df.drop(columns=[TARGET_COL])
    if dummies:
        X = pd.get_dummies(X, columns=CATEGORICAL_COLS, drop_first=True)
    return X, y

X_train, y_train = prepare_data(df_train)
X_test,  y_test  = prepare_data(df_test)

print(f"Features: {X_train.shape[1]}")

Features: 24


## Experimento MLflow

In [5]:
EXPERIMENT_NAME = "airline-satisfaction-hyperparam-search"

if not mlflow.get_experiment_by_name(EXPERIMENT_NAME):
    mlflow.create_experiment(EXPERIMENT_NAME)

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experimento: {EXPERIMENT_NAME}")

Experimento: airline-satisfaction-hyperparam-search


## 1. Logistic Regression (baseline)

In [6]:
NUMERIC_COLS  = ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]
RATING_COLS   = [c for c in X_train.columns if c not in NUMERIC_COLS]

preprocessor = ColumnTransformer([
    ("scaler",   StandardScaler(),  NUMERIC_COLS),
    ("minmax",   MinMaxScaler(),    RATING_COLS),
])

mlflow.sklearn.autolog(log_models=True, silent=True)

with mlflow.start_run(run_name="LogisticRegression", tags={"model": "logistic_regression"}):
    pipe_lr = Pipeline([
        ("pre", preprocessor),
        ("cls", LogisticRegressionCV(cv=10, max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    pipe_lr.fit(X_train, y_train)

    # Métricas en test
    y_pred = pipe_lr.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["not satisfied", "satisfied"]))

mlflow.sklearn.autolog(disable=True)

               precision    recall  f1-score   support

not satisfied       0.87      0.90      0.89     14528
    satisfied       0.87      0.83      0.85     11365

     accuracy                           0.87     25893
    macro avg       0.87      0.87      0.87     25893
 weighted avg       0.87      0.87      0.87     25893

🏃 View run LogisticRegression at: http://mlflow-proxy:5001/#/experiments/2/runs/ca543697b35e48b39a409111756e5e7f
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2


## 2. K-Nearest Neighbors + Optuna

In [7]:
preprocessor_knn = Pipeline([
    ("pre", ColumnTransformer([
        ("scaler", StandardScaler(),  NUMERIC_COLS),
        ("minmax", MinMaxScaler(),    RATING_COLS),
    ]))
])

X_train_knn = preprocessor_knn.fit_transform(X_train)
X_test_knn  = preprocessor_knn.transform(X_test)

with mlflow.start_run(run_name="KNN", tags={"model": "knn"}) as parent_run:

    def knn_objective(trial):
        params = {
            "n_neighbors": trial.suggest_int("n_neighbors", 1, 50),
            "weights":     trial.suggest_categorical("weights", ["uniform", "distance"]),
            "p":           trial.suggest_int("p", 1, 2),  # 1=Manhattan, 2=Euclidean — float causa brute force
        }
        with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
            mlflow.log_params(params)
            model = KNeighborsClassifier(**params, n_jobs=-1)
            score = cross_val_score(model, X_train_knn, y_train, scoring="f1", cv=3, n_jobs=-1).mean()
            mlflow.log_metric("f1_cv", score)
        return score

    study_knn = optuna.create_study(direction="maximize")
    study_knn.optimize(knn_objective, n_trials=5)

    mlflow.log_params(study_knn.best_params)
    mlflow.log_metric("best_f1", study_knn.best_value)

    best_knn = KNeighborsClassifier(**study_knn.best_params, n_jobs=-1).fit(X_train_knn, y_train)
    mlflow.sklearn.log_model(best_knn, "model")

    y_pred = best_knn.predict(X_test_knn)
    print(f"KNN — Best F1 (CV): {study_knn.best_value:.4f}")
    print(f"Best params: {study_knn.best_params}")
    print(classification_report(y_test, y_pred, target_names=["not satisfied", "satisfied"]))

🏃 View run trial_0 at: http://mlflow-proxy:5001/#/experiments/2/runs/57e0019e5ee24940850d2f78ade2865f
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_1 at: http://mlflow-proxy:5001/#/experiments/2/runs/ecd08439bb9a4064b11f95bfcb343144
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_2 at: http://mlflow-proxy:5001/#/experiments/2/runs/3921ce61c7ff419aad94eee4adddc58f
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_3 at: http://mlflow-proxy:5001/#/experiments/2/runs/65fb3ee6081244659fe1695460d0d588
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2


2026/04/11 00:06:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run trial_4 at: http://mlflow-proxy:5001/#/experiments/2/runs/53d8bf726d374de9af5a34bf0e4724f4
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2


2026/04/11 00:06:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


KNN — Best F1 (CV): 0.9065
Best params: {'n_neighbors': 38, 'weights': 'distance', 'p': 1}
               precision    recall  f1-score   support

not satisfied       0.92      0.95      0.93     14528
    satisfied       0.93      0.89      0.91     11365

     accuracy                           0.92     25893
    macro avg       0.92      0.92      0.92     25893
 weighted avg       0.92      0.92      0.92     25893

🏃 View run KNN at: http://mlflow-proxy:5001/#/experiments/2/runs/471132a2606b4b12b55514e34a9be68d
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2


## 3. Random Forest + Optuna

In [8]:
with mlflow.start_run(run_name="RandomForest", tags={"model": "random_forest"}) as parent_run:

    def rf_objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 50, 400),
            "criterion":         trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "max_depth":         trial.suggest_int("max_depth", 3, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 20),
        }
        with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
            mlflow.log_params(params)
            model = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
            score = cross_val_score(model, X_train, y_train, scoring="f1", cv=3, n_jobs=-1).mean()
            mlflow.log_metric("f1_cv", score)
        return score

    study_rf = optuna.create_study(direction="maximize")
    study_rf.optimize(rf_objective, n_trials=5)

    mlflow.log_params(study_rf.best_params)
    mlflow.log_metric("best_f1", study_rf.best_value)

    best_rf = RandomForestClassifier(**study_rf.best_params, random_state=RANDOM_STATE, n_jobs=-1)
    best_rf.fit(X_train, y_train)
    mlflow.sklearn.log_model(best_rf, "model")

    y_pred = best_rf.predict(X_test)
    print(f"Random Forest — Best F1 (CV): {study_rf.best_value:.4f}")
    print(f"Best params: {study_rf.best_params}")
    print(classification_report(y_test, y_pred, target_names=["not satisfied", "satisfied"]))

🏃 View run trial_0 at: http://mlflow-proxy:5001/#/experiments/2/runs/3f5c2a6f776845f2b4ddf1fc89a065b2
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_1 at: http://mlflow-proxy:5001/#/experiments/2/runs/15470d76070b475fb32ad64ae5cb399d
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_2 at: http://mlflow-proxy:5001/#/experiments/2/runs/5f38d0b157af4e99ba332d0fad279647
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_3 at: http://mlflow-proxy:5001/#/experiments/2/runs/0377be294e3746fd881b7c0de9159ee2
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_4 at: http://mlflow-proxy:5001/#/experiments/2/runs/e50d48085fc24eebbcaf2d111e251071
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2


2026/04/11 00:07:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 00:07:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest — Best F1 (CV): 0.9534
Best params: {'n_estimators': 172, 'criterion': 'entropy', 'max_depth': 23, 'min_samples_split': 13, 'min_samples_leaf': 3}
               precision    recall  f1-score   support

not satisfied       0.96      0.98      0.97     14528
    satisfied       0.97      0.94      0.96     11365

     accuracy                           0.96     25893
    macro avg       0.96      0.96      0.96     25893
 weighted avg       0.96      0.96      0.96     25893

🏃 View run RandomForest at: http://mlflow-proxy:5001/#/experiments/2/runs/49915ecfd8c54d7589f843448415f54a
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2


## 4. XGBoost + Optuna

In [9]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=RANDOM_STATE)

with mlflow.start_run(run_name="XGBoost", tags={"model": "xgboost"}) as parent_run:

    def xgb_objective(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 50, 400),
            "max_depth":        trial.suggest_int("max_depth", 2, 10),
            "grow_policy":      trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
            "learning_rate":    trial.suggest_float("learning_rate", 1e-4, 1.0, log=True),
            "gamma":            trial.suggest_float("gamma", 1e-8, 1.0, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
            "objective":        "binary:logistic",
            "eval_metric":      "auc",
            "device":           "cpu",
            "tree_method":      "hist",
            "random_state":     RANDOM_STATE,
        }
        pruning_cb = optuna.integration.XGBoostPruningCallback(trial, "validation_0-auc")

        with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
            log_params = {k: v for k, v in params.items() if k not in ["objective", "eval_metric", "device", "tree_method"]}
            mlflow.log_params(log_params)

            model = XGBClassifier(**params, callbacks=[pruning_cb], early_stopping_rounds=20, verbosity=0)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

            y_pred_val = model.predict(X_val)
            from sklearn.metrics import f1_score
            score = f1_score(y_val, y_pred_val)
            mlflow.log_metric("f1_cv", score)
        return score

    study_xgb = optuna.create_study(
        direction="maximize",
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    )
    study_xgb.optimize(xgb_objective, n_trials=5)

    mlflow.log_params(study_xgb.best_params)
    mlflow.log_metric("best_f1", study_xgb.best_value)

    best_xgb_params = {**study_xgb.best_params, "objective": "binary:logistic", "device": "cpu", "tree_method": "hist"}
    best_xgb = XGBClassifier(**best_xgb_params, random_state=RANDOM_STATE, verbosity=0)
    best_xgb.fit(X_train, y_train)
    mlflow.xgboost.log_model(best_xgb, "model")

    y_pred = best_xgb.predict(X_test)
    print(f"XGBoost — Best F1 (val): {study_xgb.best_value:.4f}")
    print(f"Best params: {study_xgb.best_params}")
    print(classification_report(y_test, y_pred, target_names=["not satisfied", "satisfied"]))

🏃 View run trial_0 at: http://mlflow-proxy:5001/#/experiments/2/runs/23445833e53949af94555cbc30bc20d2
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_1 at: http://mlflow-proxy:5001/#/experiments/2/runs/9a378ced05be4dd4b4e99f5f4f1f4717
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_2 at: http://mlflow-proxy:5001/#/experiments/2/runs/6dbdfd5feacc4536ac32d0cd43530dde
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_3 at: http://mlflow-proxy:5001/#/experiments/2/runs/3d6c8ea199e74b789b4536c7bccf886b
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2
🏃 View run trial_4 at: http://mlflow-proxy:5001/#/experiments/2/runs/7ff54a1ef0b04afa855bd8e3ac3bd5d8
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2


2026/04/11 00:08:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost — Best F1 (val): 0.9467
Best params: {'n_estimators': 364, 'max_depth': 3, 'grow_policy': 'depthwise', 'learning_rate': 0.06146059145138084, 'gamma': 4.558158449337475e-08, 'min_child_weight': 4, 'subsample': 0.955952889168917, 'colsample_bytree': 0.6478660452796294, 'reg_alpha': 0.3092154935865513, 'reg_lambda': 8.529248612266795e-07}
               precision    recall  f1-score   support

not satisfied       0.95      0.97      0.96     14528
    satisfied       0.96      0.94      0.95     11365

     accuracy                           0.95     25893
    macro avg       0.95      0.95      0.95     25893
 weighted avg       0.95      0.95      0.95     25893

🏃 View run XGBoost at: http://mlflow-proxy:5001/#/experiments/2/runs/ef319e186ddf4823b2f9fbb22fb845c9
🧪 View experiment at: http://mlflow-proxy:5001/#/experiments/2


## Comparación de los modelos, tomandolos de MLFlow

In [10]:
client = mlflow.MlflowClient()
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.model LIKE '%'",
    order_by=["metrics.best_f1 DESC"],
)

best_per_model = {}
for run in runs:
    name = run.data.tags.get("model", "unknown")
    f1   = run.data.metrics.get("best_f1", float("nan"))
    if math.isnan(f1):
        continue
    if name not in best_per_model or f1 > best_per_model[name]["f1"]:
        best_per_model[name] = {"f1": f1, "run": run}

print("Resultados por modelo:")
print(f"{'Modelo':<25} {'Best F1':>10}")
print("-" * 37)
for name, data in sorted(best_per_model.items(), key=lambda x: -x[1]["f1"]):
    print(f"{name:<25} {data['f1']:>10.4f}")

winner_name, winner_data = max(best_per_model.items(), key=lambda x: x[1]["f1"])
best_run = winner_data["run"]
print(f"\nMejor modelo: {winner_name} — F1: {winner_data['f1']:.4f}")

# asi lo registro:
#model_uri = f"runs:/{best_run.info.run_id}/model"
#registered = mlflow.register_model(model_uri, "airline-satisfaction-best")
#print(f"Registrado: {registered.name} v{registered.version}")


Resultados por modelo:
Modelo                       Best F1
-------------------------------------
random_forest                 0.9534
xgboost                       0.9467
knn                           0.9065

Mejor modelo: random_forest — F1: 0.9534
